In [4]:
!pip install -q opencv-contrib-python numpy matplotlib

In [5]:
import os
import re
import zipfile
import cv2
import numpy as np
import random
import math
import csv
import matplotlib.pyplot as plt
from IPython.display import display, Image

In [6]:
from google.colab import files
uploaded = files.upload()

In [7]:
image_folder = "/content"

images = sorted([f for f in os.listdir(image_folder) if f.endswith(".jpg")])

print("Total images:", len(images))
print(images[:5])

Total images: 19
['oyla_0001.jpg', 'oyla_0003.jpg', 'oyla_0005.jpg', 'oyla_0007.jpg', 'oyla_0009.jpg']


In [8]:
sift = cv2.SIFT_create()

descriptors = {}
keypoints = {}

for img_name in images:
    img_path = os.path.join(image_folder, img_name)
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    kp, des = sift.detectAndCompute(img, None)

    keypoints[img_name] = kp
    descriptors[img_name] = des

    print(img_name, "keypoints:", len(kp))

oyla_0001.jpg keypoints: 17538
oyla_0003.jpg keypoints: 17608
oyla_0005.jpg keypoints: 19553
oyla_0007.jpg keypoints: 20960
oyla_0009.jpg keypoints: 22456
oyla_0011.jpg keypoints: 21016
oyla_0013.jpg keypoints: 19459
oyla_0015.jpg keypoints: 19336
oyla_0017.jpg keypoints: 21478
oyla_0019.jpg keypoints: 18460
oyla_0021.jpg keypoints: 19223
oyla_0023.jpg keypoints: 14956
oyla_0025.jpg keypoints: 12319
oyla_0027.jpg keypoints: 10794
oyla_0029.jpg keypoints: 9439
oyla_0031.jpg keypoints: 11947
oyla_0033.jpg keypoints: 12541
oyla_0035.jpg keypoints: 12733
oyla_0037.jpg keypoints: 12056


In [9]:
bf = cv2.BFMatcher()

def compute_match_cost(img1, img2):

    des1 = descriptors[img1]
    des2 = descriptors[img2]

    matches = bf.knnMatch(des1, des2, k=2)

    good_matches = []

    for m, n in matches:
        if m.distance < 0.75 * n.distance:   # Lowe ratio test
            good_matches.append(m)

    total_cost = sum(m.distance for m in good_matches)

    return total_cost, len(good_matches)

In [10]:
# Fix: step by 1 (not 2), and add the wrap-around pair (0037 -> 0001)
consecutive_costs = []
consecutive_pairs = []

for i in range(len(images)):
    img1 = images[i]
    img2 = images[(i + 1) % len(images)]  # wraps around at end
    consecutive_pairs.append((img1, img2))
    cost, matches = compute_match_cost(img1, img2)
    consecutive_costs.append(cost)
    print(f"{img1} vs {img2} | matches: {matches} | cost: {cost:.2f}")

print(f"\nTotal consecutive pairs: {len(consecutive_costs)}")
print(f"Average consecutive cost: {np.mean(consecutive_costs):.2f}")

oyla_0001.jpg vs oyla_0003.jpg | matches: 6245 | cost: 888675.83
oyla_0003.jpg vs oyla_0005.jpg | matches: 6736 | cost: 906768.09
oyla_0005.jpg vs oyla_0007.jpg | matches: 6228 | cost: 927506.57
oyla_0007.jpg vs oyla_0009.jpg | matches: 5101 | cost: 794591.09
oyla_0009.jpg vs oyla_0011.jpg | matches: 8466 | cost: 1189380.00
oyla_0011.jpg vs oyla_0013.jpg | matches: 5455 | cost: 841990.95
oyla_0013.jpg vs oyla_0015.jpg | matches: 12137 | cost: 961915.74
oyla_0015.jpg vs oyla_0017.jpg | matches: 1998 | cost: 336660.80
oyla_0017.jpg vs oyla_0019.jpg | matches: 3359 | cost: 558623.61
oyla_0019.jpg vs oyla_0021.jpg | matches: 2838 | cost: 473061.86
oyla_0021.jpg vs oyla_0023.jpg | matches: 2685 | cost: 454176.00
oyla_0023.jpg vs oyla_0025.jpg | matches: 2518 | cost: 402086.77
oyla_0025.jpg vs oyla_0027.jpg | matches: 1847 | cost: 297519.84
oyla_0027.jpg vs oyla_0029.jpg | matches: 1690 | cost: 256304.03
oyla_0029.jpg vs oyla_0031.jpg | matches: 1169 | cost: 173716.22
oyla_0031.jpg vs oyla_0

In [11]:
import random

random_costs = []
random_pairs = []

# Build a set of consecutive pairs to exclude them
consecutive_set = set(consecutive_pairs) | {(b, a) for a, b in consecutive_pairs}

attempts = 0
while len(random_pairs) < 19 and attempts < 10000:
    a, b = random.sample(images, 2)
    if (a, b) not in consecutive_set and (a, b) not in random_pairs and (b, a) not in random_pairs:
        random_pairs.append((a, b))
    attempts += 1

for img1, img2 in random_pairs:
    cost, matches = compute_match_cost(img1, img2)
    random_costs.append(cost)
    print(f"{img1} vs {img2} | matches: {matches} | cost: {cost:.2f}")

print(f"\nTotal random pairs: {len(random_costs)}")
print(f"Average random cost: {np.mean(random_costs):.2f}")

oyla_0033.jpg vs oyla_0015.jpg | matches: 102 | cost: 18721.38
oyla_0003.jpg vs oyla_0027.jpg | matches: 156 | cost: 26086.83
oyla_0025.jpg vs oyla_0031.jpg | matches: 188 | cost: 33278.46
oyla_0025.jpg vs oyla_0021.jpg | matches: 289 | cost: 50941.41
oyla_0013.jpg vs oyla_0009.jpg | matches: 1841 | cost: 306547.53
oyla_0019.jpg vs oyla_0027.jpg | matches: 98 | cost: 17898.82
oyla_0027.jpg vs oyla_0037.jpg | matches: 101 | cost: 17536.25
oyla_0017.jpg vs oyla_0009.jpg | matches: 200 | cost: 33359.48
oyla_0017.jpg vs oyla_0005.jpg | matches: 83 | cost: 14850.96
oyla_0029.jpg vs oyla_0005.jpg | matches: 139 | cost: 21707.70
oyla_0037.jpg vs oyla_0007.jpg | matches: 95 | cost: 16042.46
oyla_0015.jpg vs oyla_0005.jpg | matches: 245 | cost: 42941.37
oyla_0023.jpg vs oyla_0001.jpg | matches: 72 | cost: 12912.68
oyla_0037.jpg vs oyla_0003.jpg | matches: 64 | cost: 11246.30
oyla_0001.jpg vs oyla_0005.jpg | matches: 2583 | cost: 393319.67
oyla_0029.jpg vs oyla_0013.jpg | matches: 54 | cost: 923

In [12]:
avg_consec = np.mean(consecutive_costs)
avg_random = np.mean(random_costs)

print(f"Average cost (consecutive): {avg_consec:.2f}")
print(f"Average cost (random):      {avg_random:.2f}")

if avg_consec > avg_random:
    print("\nResult: Consecutive cost is HIGHER than random.")
else:
    print("\nResult: Consecutive cost is LOWER than random.")

print("""
Answer to Q5:
Consecutive pairs should have HIGHER matching cost than random pairs.

Why? SIFT match cost = sum of feature distances for all GOOD matches.
Consecutive images share much more visual overlap (camera moves slightly),
so they produce many more good matches (high match count).
Even though each individual match may be of similar quality,
the sheer volume of matches in consecutive pairs drives the total cost UP.

Random pairs share little visual overlap, so very few features match,
meaning the total summed cost stays LOW -- not because matches are worse,
but because there are far fewer of them.

Takeaway: Higher cost here means MORE shared content, not worse matching.
""")

Average cost (consecutive): 542214.29
Average cost (random):      57291.28

Result: Consecutive cost is HIGHER than random.

Answer to Q5:
Consecutive pairs should have HIGHER matching cost than random pairs.

Why? SIFT match cost = sum of feature distances for all GOOD matches.
Consecutive images share much more visual overlap (camera moves slightly),
so they produce many more good matches (high match count).
Even though each individual match may be of similar quality,
the sheer volume of matches in consecutive pairs drives the total cost UP.

Random pairs share little visual overlap, so very few features match,
meaning the total summed cost stays LOW -- not because matches are worse,
but because there are far fewer of them.

Takeaway: Higher cost here means MORE shared content, not worse matching.

